# 🐼 Pandas for Data Analysis
## Industry-Level Reference & Revision Guide
### EDA · Data Cleaning · GroupBy · Merging · DateTime · String Ops

---
> **How to use this notebook:** Every section has code first → output → logic explanation. Run each cell, read the output, then read the explanation below it.

---

In [2]:
# ── IMPORTS ──────────────────────────────────────────────
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 20)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print('Libraries loaded successfully!')

Libraries loaded successfully!


---
# 📦 SECTION 1 — Data Loading & First Look
**Dataset: Titanic** (891 rows, 15 columns) — passenger survival data.

| Column | Description |
|--------|-------------|
| survived | 0 = died, 1 = survived |
| pclass | Ticket class (1, 2, 3) |
| sex | male / female |
| age | Age in years |
| fare | Ticket price |
| embarked | Port: S=Southampton, C=Cherbourg, Q=Queenstown |
| deck | Cabin deck (lots of nulls) |

In [3]:
# ── 1.1 LOADING DATA ─────────────────────────────────────

# Load full dataset
df = pd.read_csv('titanic.csv')

# Load specific columns only — saves memory on large files
df_small = pd.read_csv('titanic.csv', usecols=['survived','pclass','sex','age','fare'])

# Load with row limit — for quick inspection of huge files
df_100 = pd.read_csv('titanic.csv', nrows=100)

print('Full shape:', df.shape)
print('Selected cols shape:', df_small.shape)
print('100 rows shape:', df_100.shape)

Full shape: (891, 15)
Selected cols shape: (891, 5)
100 rows shape: (100, 15)


> **LOGIC:** `usecols` is critical in industry. Real datasets have 100+ columns. Load only what you need — saves memory and speeds up processing.

In [4]:
# ── 1.2 FIRST INSPECTION ─────────────────────────────────

print('=== SHAPE ===')
print(df.shape)

print('\n=== FIRST 5 ROWS ===')
print(df.head())

print('\n=== LAST 5 ROWS ===')
print(df.tail())

print('\n=== RANDOM SAMPLE (unbiased view) ===')
print(df.sample(5, random_state=42))

=== SHAPE ===
(891, 15)

=== FIRST 5 ROWS ===
   survived  pclass     sex   age  sibsp  parch  fare embarked  class    who  \
0         0       3    male 22.00      1      0  7.25        S  Third    man   
1         1       1  female 38.00      1      0 71.28        C  First  woman   
2         1       3  female 26.00      0      0  7.92        S  Third  woman   
3         1       1  female 35.00      1      0 53.10        S  First  woman   
4         0       3    male 35.00      0      0  8.05        S  Third    man   

   adult_male deck  embark_town alive  alone  
0        True  NaN  Southampton    no  False  
1       False    C    Cherbourg   yes  False  
2       False  NaN  Southampton   yes   True  
3       False    C  Southampton   yes  False  
4        True  NaN  Southampton    no   True  

=== LAST 5 ROWS ===
     survived  pclass     sex   age  sibsp  parch  fare embarked   class  \
886         0       2    male 27.00      0      0 13.00        S  Second   
887         1     

In [5]:
# ── 1.3 DATA TYPES + NULL COUNTS + MEMORY ────────────────

print('=== INFO ===')
df.info()

print('\n=== COLUMN NAMES ===')
print(df.columns.tolist())

print('\n=== DTYPES ===')
print(df.dtypes)

=== INFO ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   survived     891 non-null    int64  
 1   pclass       891 non-null    int64  
 2   sex          891 non-null    object 
 3   age          714 non-null    float64
 4   sibsp        891 non-null    int64  
 5   parch        891 non-null    int64  
 6   fare         891 non-null    float64
 7   embarked     889 non-null    object 
 8   class        891 non-null    object 
 9   who          891 non-null    object 
 10  adult_male   891 non-null    bool   
 11  deck         203 non-null    object 
 12  embark_town  889 non-null    object 
 13  alive        891 non-null    object 
 14  alone        891 non-null    bool   
dtypes: bool(2), float64(2), int64(4), object(7)
memory usage: 92.4+ KB

=== COLUMN NAMES ===
['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked',

In [6]:
# ── 1.4 STATISTICAL SUMMARY ──────────────────────────────

print('=== NUMERICAL COLUMNS ===')
print(df.describe())

print('\n=== CATEGORICAL COLUMNS ===')
print(df.describe(include=['object']))

print('\n=== ALL COLUMNS ===')
print(df.describe(include='all'))

=== NUMERICAL COLUMNS ===
       survived  pclass    age  sibsp  parch   fare
count    891.00  891.00 714.00 891.00 891.00 891.00
mean       0.38    2.31  29.70   0.52   0.38  32.20
std        0.49    0.84  14.53   1.10   0.81  49.69
min        0.00    1.00   0.42   0.00   0.00   0.00
25%        0.00    2.00  20.12   0.00   0.00   7.91
50%        0.00    3.00  28.00   0.00   0.00  14.45
75%        1.00    3.00  38.00   1.00   0.00  31.00
max        1.00    3.00  80.00   8.00   6.00 512.33

=== CATEGORICAL COLUMNS ===
         sex embarked  class  who deck  embark_town alive
count    891      889    891  891  203          889   891
unique     2        3      3    3    7            3     2
top     male        S  Third  man    C  Southampton    no
freq     577      644    491  537   59          644   549

=== ALL COLUMNS ===
        survived  pclass   sex    age  sibsp  parch   fare embarked  class  \
count     891.00  891.00   891 714.00 891.00 891.00 891.00      889    891   
unique    

> **LOGIC:** Always run `head()` → `info()` → `describe()` as your first 3 steps on any new dataset. This gives you the full picture before touching anything.

---
# 🧹 SECTION 2 — Data Cleaning
Real data is always messy. This is 60–70% of a data analyst's actual job.

In [7]:
# ── 2.1 FINDING NULL VALUES ──────────────────────────────

df = pd.read_csv('titanic.csv')  # fresh load

print('=== NULL COUNT PER COLUMN ===')
print(df.isnull().sum())

print('\n=== NULL PERCENTAGE (sorted) ===')
null_pct = (df.isnull().sum() / len(df)) * 100
print(null_pct[null_pct > 0].sort_values(ascending=False).round(1))

=== NULL COUNT PER COLUMN ===
survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64

=== NULL PERCENTAGE (sorted) ===
deck          77.20
age           19.90
embarked       0.20
embark_town    0.20
dtype: float64


> **LOGIC:** `isnull()` returns a DataFrame of True/False. `.sum()` counts Trues per column (True=1, False=0). Dividing by `len(df)` converts to proportion, multiply by 100 for percentage.

In [8]:
# ── 2.2 HANDLING NULL VALUES ─────────────────────────────

df = pd.read_csv('titanic.csv')

print('Shape before cleaning:', df.shape)

# Drop rows where age is null only — targeted drop
# df.dropna(subset=['age'], inplace=True)

# Drop columns with more than 50% nulls
threshold = len(df) * 0.5
df.dropna(thresh=threshold, axis=1, inplace=True)
print('Shape after dropping high-null columns:', df.shape)

# Fill nulls — different strategy per column
df.fillna({
    'age':      df['age'].median(),             # numerical -> median
    'fare':     df['fare'].mean(),              # numerical -> mean
    'embarked': df['embarked'].mode()[0],       # categorical -> mode
}, inplace=True)

print('\nNull count after filling:')
print(df.isnull().sum()[df.isnull().sum() > 0])

Shape before cleaning: (891, 15)
Shape after dropping high-null columns: (891, 14)

Null count after filling:
embark_town    2
dtype: int64


> **LOGIC:** Never blindly drop nulls — you lose data. Strategy:
> - Numerical columns → use **median** (robust to outliers) or **mean** (for normal distributions)
> - Categorical columns → use **mode** (most frequent value)
> - >50% nulls → just **drop the column**

In [9]:
# ── 2.3 FFILL & BFILL ────────────────────────────────────

# Create sample data with nulls to demonstrate
sample = pd.DataFrame({
    'date': ['2024-01-01','2024-01-02','2024-01-03','2024-01-04','2024-01-05'],
    'price': [100, None, None, 130, None]
})

print('=== ORIGINAL ===')
print(sample)

print('\n=== AFTER FFILL (carry forward) ===')
print(sample.fillna(method='ffill'))

print('\n=== AFTER BFILL (carry backward) ===')
print(sample.fillna(method='bfill'))

=== ORIGINAL ===
         date  price
0  2024-01-01 100.00
1  2024-01-02    NaN
2  2024-01-03    NaN
3  2024-01-04 130.00
4  2024-01-05    NaN

=== AFTER FFILL (carry forward) ===
         date  price
0  2024-01-01 100.00
1  2024-01-02 100.00
2  2024-01-03 100.00
3  2024-01-04 130.00
4  2024-01-05 130.00

=== AFTER BFILL (carry backward) ===
         date  price
0  2024-01-01 100.00
1  2024-01-02 130.00
2  2024-01-03 130.00
3  2024-01-04 130.00
4  2024-01-05    NaN


C:\Users\Naman\AppData\Local\Temp\ipykernel_13556\2535401973.py:13: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  print(sample.fillna(method='ffill'))
C:\Users\Naman\AppData\Local\Temp\ipykernel_13556\2535401973.py:16: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  print(sample.fillna(method='bfill'))


> **LOGIC:** `ffill` = forward fill → copies the value from the row above. `bfill` = backward fill → copies from the row below.
> - **ffill edge case:** If the first row is NaN, ffill cannot fill it — nothing above.
> - **bfill edge case:** If the last row is NaN, bfill cannot fill it — nothing below.
> - **Industry use:** Stock prices on weekends, sensor readings with gaps.

In [10]:
# ── 2.4 TYPECASTING ──────────────────────────────────────

df = pd.read_csv('titanic.csv')

print('=== BEFORE TYPECASTING ===')
print(df[['survived','pclass','sex']].dtypes)
print('Memory:', df.memory_usage(deep=True).sum() // 1024, 'KB')

# Fix types
df['survived'] = df['survived'].astype(bool)        # 0/1 -> True/False
df['pclass']   = df['pclass'].astype('category')    # saves memory
df['sex']      = df['sex'].astype('category')       # limited unique values

print('\n=== AFTER TYPECASTING ===')
print(df[['survived','pclass','sex']].dtypes)
print('Memory:', df.memory_usage(deep=True).sum() // 1024, 'KB')

# Find all object columns at once
obj_cols = df.select_dtypes(include=['object']).columns.tolist()
num_cols = df.select_dtypes(include=['int64','float64']).columns.tolist()
print('\nString columns:', obj_cols)
print('Numeric columns:', num_cols)

=== BEFORE TYPECASTING ===
survived     int64
pclass       int64
sex         object
dtype: object
Memory: 354 KB

=== AFTER TYPECASTING ===
survived        bool
pclass      category
sex         category
dtype: object
Memory: 297 KB

String columns: ['embarked', 'class', 'who', 'deck', 'embark_town', 'alive']
Numeric columns: ['age', 'sibsp', 'parch', 'fare']


> **LOGIC:** Converting repeated string columns (sex, class, embarked) to `category` type reduces memory by 50–90% on large datasets. Critical for production pipelines with millions of rows.

---
# ⚙️ SECTION 3 — Column Operations

In [11]:
# ── 3.1 CREATING NEW COLUMNS (apply + lambda) ────────────

df = pd.read_csv('titanic.csv')
df['age'].fillna(df['age'].median(), inplace=True)

# Simple arithmetic column
df['fare_per_class'] = df['fare'] / df['pclass']

# Age group — nested lambda
df['age_group'] = df['age'].apply(lambda x:
    'Child' if x < 18 else ('Senior' if x >= 60 else 'Adult'))

# Family size
df['family_size'] = df['sibsp'] + df['parch'] + 1

# Is alone
df['is_alone'] = df['family_size'].apply(lambda x: True if x == 1 else False)

# Fare tier using pd.cut — cleaner way to bin numerical column
df['fare_tier'] = pd.cut(df['fare'], bins=[0,10,50,600], labels=['Low','Mid','High'])

print(df[['age','age_group','family_size','is_alone','fare_tier']].head(8))

    age age_group  family_size  is_alone fare_tier
0 22.00     Adult            2     False       Low
1 38.00     Adult            2     False      High
2 26.00     Adult            1      True       Low
3 35.00     Adult            2     False      High
4 35.00     Adult            1      True       Low
5 28.00     Adult            1      True       Low
6 54.00     Adult            1      True      High
7  2.00     Child            5     False       Mid


C:\Users\Naman\AppData\Local\Temp\ipykernel_13556\389232194.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['age'].fillna(df['age'].median(), inplace=True)


> **LOGIC:** `apply(lambda x: ...)` — for every value in the column, call it `x` and apply the logic.
> - `pd.cut()` is better than lambda for binning — it handles edges and is faster.
> - `pd.cut(col, bins=[...], labels=[...])` — bins define the ranges, labels define the names.

In [12]:
# ── 3.2 STRING OPERATIONS ────────────────────────────────

data = pd.DataFrame({'text': [
    'Hello Data Science',
    'I love Machine Learning',
    'I read Data Science books',
    '  extra spaces  ',
]})

# .str. accessor — apply string methods on entire column
data['lower']      = data['text'].str.lower()
data['upper']      = data['text'].str.upper()
data['stripped']   = data['text'].str.strip()           # remove spaces
data['has_data']   = data['text'].str.contains('Data', na=False)
data['word_count'] = data['text'].apply(lambda x: len(str(x).strip().split()))
data['starts_H']   = data['text'].str.startswith('H')
data['length']     = data['text'].str.len()

print(data)

                        text                      lower  \
0         Hello Data Science         hello data science   
1    I love Machine Learning    i love machine learning   
2  I read Data Science books  i read data science books   
3             extra spaces               extra spaces     

                       upper                   stripped  has_data  word_count  \
0         HELLO DATA SCIENCE         Hello Data Science      True           3   
1    I LOVE MACHINE LEARNING    I love Machine Learning     False           4   
2  I READ DATA SCIENCE BOOKS  I read Data Science books      True           5   
3             EXTRA SPACES                 extra spaces     False           2   

   starts_H  length  
0      True      18  
1     False      23  
2     False      25  
3     False      16  


> **LOGIC:** `.str.` tells pandas the column contains strings — apply this method on all values at once.
> - `apply(lambda x: len(x.split()))` for word count because it combines two operations — `.split()` then `len()`.
> - `.str.strip()` is always used after `.str.lower()` in real data — extra spaces cause join mismatches.
> - `na=False` in `.str.contains()` handles NaN values — without it, NaN rows throw an error.

---
# 🔍 SECTION 4 — Filtering & Boolean Indexing

In [13]:
# ── 4.1 SINGLE & MULTI-CONDITION FILTERING ───────────────

df = pd.read_csv('titanic.csv')

# Single condition
children = df[df['age'] < 18]
print('Children count:', len(children))

# AND condition — both must be True
rich_survivors = df[(df['fare'] > 100) & (df['survived'] == 1)]
print('Rich survivors:', len(rich_survivors))

# OR condition — either can be True
vip = df[(df['pclass'] == 1) | (df['fare'] > 200)]
print('VIP passengers:', len(vip))

# NOT condition
not_southampton = df[~(df['embarked'] == 'S')]
print('Not from Southampton:', len(not_southampton))

# isin — filter by multiple values
port_cq = df[df['embarked'].isin(['C', 'Q'])]
print('C and Q passengers:', len(port_cq))

# Above average fare
avg_fare = df['fare'].mean()
above_avg = df[df['fare'] > avg_fare]
print(f'Above average fare ({avg_fare:.2f}):', len(above_avg))

Children count: 113
Rich survivors: 39
VIP passengers: 216
Not from Southampton: 247
C and Q passengers: 245
Above average fare (32.20): 211


> **LOGIC:** `df[condition]` filters rows where condition is True. Each condition returns a boolean Series (True/False per row). pandas keeps only True rows.
> - Always wrap each condition in `()` when using `&` or `|`.
> - `~` = NOT operator — flips True to False and vice versa.

In [14]:
# ── 4.2 loc & iloc ───────────────────────────────────────

df = pd.read_csv('titanic.csv')

# iloc — by integer position
print('=== FIRST ROW (iloc) ===')
print(df.iloc[0])

print('\n=== LAST ROW (iloc) ===')
print(df.iloc[-1])

print('\n=== FIRST 3 ROWS, FIRST 3 COLS (iloc) ===')
print(df.iloc[0:3, 0:3])

print('\n=== SPECIFIC ROWS AND COLS (iloc) ===')
print(df.iloc[[0, 10, 20], [0, 1, 2, 3]])

# loc — by label name
print('\n=== FILTER + SELECT COLS IN ONE LINE (loc) ===')
result = df.loc[
    (df['sex'] == 'female') & (df['pclass'] == 1),
    ['age', 'fare', 'survived']
]
print(result.head(5))

# iloc vs loc slicing — important difference
print('\n=== iloc[0:3] — row 3 EXCLUDED ===')
print(df.iloc[0:3][['survived','pclass']])

print('\n=== loc[0:3] — row 3 INCLUDED ===')
print(df.loc[0:3, ['survived','pclass']])

=== FIRST ROW (iloc) ===
survived                 0
pclass                   3
sex                   male
age                  22.00
sibsp                    1
parch                    0
fare                  7.25
embarked                 S
class                Third
who                    man
adult_male            True
deck                   NaN
embark_town    Southampton
alive                   no
alone                False
Name: 0, dtype: object

=== LAST ROW (iloc) ===
survived                0
pclass                  3
sex                  male
age                 32.00
sibsp                   0
parch                   0
fare                 7.75
embarked                Q
class               Third
who                   man
adult_male           True
deck                  NaN
embark_town    Queenstown
alive                  no
alone                True
Name: 890, dtype: object

=== FIRST 3 ROWS, FIRST 3 COLS (iloc) ===
   survived  pclass     sex
0         0       3    male
1       

> **LOGIC:**
> - `iloc` = integer location → position numbers → end excluded (like Python slicing)
> - `loc` = label location → column/row names → end included
> - `df.loc[condition, columns]` is the industry standard — filter rows AND select columns in one line. Avoids chained indexing warning.

---
# 📊 SECTION 5 — GroupBy & Aggregation

In [15]:
# ── 5.1 BASIC GROUPBY ────────────────────────────────────

df = pd.read_csv('titanic.csv')

print('=== SURVIVAL RATE BY SEX ===')
print((df.groupby('sex')['survived'].mean() * 100).round(1))

print('\n=== AVERAGE FARE BY CLASS ===')
print(df.groupby('pclass')['fare'].mean().round(2))

print('\n=== MULTIPLE STATS ON ONE COLUMN ===')
print(df.groupby('pclass')['age'].agg(['mean','median','min','max']).round(2))

print('\n=== MULTIPLE COLUMNS, DIFFERENT AGGREGATIONS ===')
result = df.groupby('pclass').agg({
    'survived': 'mean',
    'fare':     ['mean', 'max'],
    'age':      'median',
    'sex':      'count'
}).round(2)
print(result)

=== SURVIVAL RATE BY SEX ===
sex
female   74.20
male     18.90
Name: survived, dtype: float64

=== AVERAGE FARE BY CLASS ===
pclass
1   84.15
2   20.66
3   13.68
Name: fare, dtype: float64

=== MULTIPLE STATS ON ONE COLUMN ===
        mean  median  min   max
pclass                         
1      38.23   37.00 0.92 80.00
2      29.88   29.00 0.67 70.00
3      25.14   24.00 0.42 74.00

=== MULTIPLE COLUMNS, DIFFERENT AGGREGATIONS ===
       survived  fare           age   sex
           mean  mean    max median count
pclass                                   
1          0.63 84.15 512.33  37.00   216
2          0.47 20.66  73.50  29.00   184
3          0.24 13.68  69.55  24.00   491


> **LOGIC:** GroupBy = Split → Apply → Combine
> 1. **Split:** `groupby('pclass')` splits DataFrame into 3 mini-tables (class 1, 2, 3)
> 2. **Apply:** `.mean()` / `.agg()` calculates stats on each mini-table
> 3. **Combine:** Results merged back into one table
>
> `groupby()['survived'].mean()` = survival rate because survived is 0/1 — mean of 0s and 1s = proportion.

In [16]:
# ── 5.2 GROUPBY ON MULTIPLE COLUMNS ──────────────────────

df = pd.read_csv('titanic.csv')

print('=== SURVIVAL STATS BY SEX AND CLASS ===')
result = df.groupby(['sex','pclass'])['survived'].agg(['sum','count','mean'])
result.columns = ['survived_count','total','survival_rate']
result['survival_rate'] = (result['survival_rate'] * 100).round(1)
print(result)

print('\n=== CONVERT TO CLEAN DATAFRAME ===')
df_grouped = df.groupby(['sex','pclass','survived']).size().reset_index(name='count')
print(df_grouped)

print('\n=== SIZE vs COUNT DIFFERENCE ===')
print('size()  — counts all rows including NaN:', df.groupby('pclass').size().to_dict())
print('count() — counts non-null per column (shows all cols):')
print(df.groupby('pclass').count()[['age','fare']].head())

=== SURVIVAL STATS BY SEX AND CLASS ===
               survived_count  total  survival_rate
sex    pclass                                      
female 1                   91     94          96.80
       2                   70     76          92.10
       3                   72    144          50.00
male   1                   45    122          36.90
       2                   17    108          15.70
       3                   47    347          13.50

=== CONVERT TO CLEAN DATAFRAME ===
       sex  pclass  survived  count
0   female       1         0      3
1   female       1         1     91
2   female       2         0      6
3   female       2         1     70
4   female       3         0     72
5   female       3         1     72
6     male       1         0     77
7     male       1         1     45
8     male       2         0     91
9     male       2         1     17
10    male       3         0    300
11    male       3         1     47

=== SIZE vs COUNT DIFFERENCE ===
size()

> **LOGIC:**
> - `size()` → counts ALL rows per group (including NaN) → returns a Series
> - `count()` → counts non-null values per column per group → returns a DataFrame
> - For row counting in groupby → always use `size()`
> - `reset_index(name='count')` → moves all index levels into columns, names the values column 'count'

---
# 🔗 SECTION 6 — Merging & Joining

In [17]:
# ── 6.1 SETUP: TWO RELATED TABLES ────────────────────────

# passengers table
passengers = pd.DataFrame({
    'passenger_id': [1, 2, 3, 4, 5],
    'name':         ['Alice','Bob','Charlie','David','Eve'],
    'ticket_id':    [101, 102, 103, 104, None]
})

# tickets table
tickets = pd.DataFrame({
    'ticket_id': [101, 102, 103, 106, 107],
    'price':     [500, 250, 150, 800, 300],
    'class':     ['First','Second','Third','First','Second']
})

print('=== PASSENGERS ===')
print(passengers)
print('\n=== TICKETS ===')
print(tickets)

=== PASSENGERS ===
   passenger_id     name  ticket_id
0             1    Alice     101.00
1             2      Bob     102.00
2             3  Charlie     103.00
3             4    David     104.00
4             5      Eve        NaN

=== TICKETS ===
   ticket_id  price   class
0        101    500   First
1        102    250  Second
2        103    150   Third
3        106    800   First
4        107    300  Second


In [18]:
# ── 6.2 ALL MERGE TYPES ───────────────────────────────────

# INNER — only rows with matching ticket_id in BOTH tables
inner = pd.merge(passengers, tickets, on='ticket_id', how='inner')
print('=== INNER MERGE (only matched) ===')
print(inner)

# LEFT — all passengers, ticket info where available
left = pd.merge(passengers, tickets, on='ticket_id', how='left')
print('\n=== LEFT MERGE (all passengers) ===')
print(left)

# RIGHT — all tickets, passenger info where available
right = pd.merge(passengers, tickets, on='ticket_id', how='right')
print('\n=== RIGHT MERGE (all tickets) ===')
print(right)

# OUTER — everything from both, NaN where no match
outer = pd.merge(passengers, tickets, on='ticket_id', how='outer')
print('\n=== OUTER MERGE (everything) ===')
print(outer)

=== INNER MERGE (only matched) ===
   passenger_id     name  ticket_id  price   class
0             1    Alice     101.00    500   First
1             2      Bob     102.00    250  Second
2             3  Charlie     103.00    150   Third

=== LEFT MERGE (all passengers) ===
   passenger_id     name  ticket_id  price   class
0             1    Alice     101.00 500.00   First
1             2      Bob     102.00 250.00  Second
2             3  Charlie     103.00 150.00   Third
3             4    David     104.00    NaN     NaN
4             5      Eve        NaN    NaN     NaN

=== RIGHT MERGE (all tickets) ===
   passenger_id     name  ticket_id  price   class
0          1.00    Alice     101.00    500   First
1          2.00      Bob     102.00    250  Second
2          3.00  Charlie     103.00    150   Third
3           NaN      NaN     106.00    800   First
4           NaN      NaN     107.00    300  Second

=== OUTER MERGE (everything) ===
   passenger_id     name  ticket_id  price 

> **LOGIC:** Think of it as a Venn diagram:
> - `inner` → only the intersection (common rows)
> - `left` → all of left + intersection
> - `right` → intersection + all of right
> - `outer` → everything from both
>
> **Industry rule:** Use LEFT when left table is your primary data and right is supplementary info.

In [19]:
# ── 6.3 MERGE ON DIFFERENT COLUMN NAMES ──────────────────

df1 = pd.DataFrame({'emp_id': [1,2,3], 'name': ['Alice','Bob','Charlie']})
df2 = pd.DataFrame({'employee_id': [1,2,4], 'salary': [50000,60000,70000]})

result = pd.merge(df1, df2, left_on='emp_id', right_on='employee_id', how='left')
print(result)

# CROSS MERGE — every combination of rows
colors = pd.DataFrame({'color': ['Red','Blue']})
sizes  = pd.DataFrame({'size':  ['S','M','L']})
products = pd.merge(colors, sizes, how='cross')
print('\n=== CROSS MERGE — all color+size combinations ===')
print(products)

   emp_id     name  employee_id   salary
0       1    Alice         1.00 50000.00
1       2      Bob         2.00 60000.00
2       3  Charlie          NaN      NaN

=== CROSS MERGE — all color+size combinations ===
  color size
0   Red    S
1   Red    M
2   Red    L
3  Blue    S
4  Blue    M
5  Blue    L


In [20]:
# ── 6.4 CONCATENATION ─────────────────────────────────────

# Simulate monthly data files
jan = pd.DataFrame({'month':['Jan']*3, 'sales':[100,200,150]})
feb = pd.DataFrame({'month':['Feb']*3, 'sales':[180,220,160]})
mar = pd.DataFrame({'month':['Mar']*3, 'sales':[200,240,190]})

# Row wise — stack vertically
full = pd.concat([jan, feb, mar], ignore_index=True)
print('=== ROW WISE CONCAT (axis=0) ===')
print(full)

# Column wise — side by side
df_a = pd.DataFrame({'Name':['Alice','Bob'], 'Age':[25,30]})
df_b = pd.DataFrame({'City':['Mumbai','Delhi'], 'Score':[88,92]})
combined = pd.concat([df_a, df_b], axis=1)
print('\n=== COLUMN WISE CONCAT (axis=1) ===')
print(combined)

=== ROW WISE CONCAT (axis=0) ===
  month  sales
0   Jan    100
1   Jan    200
2   Jan    150
3   Feb    180
4   Feb    220
5   Feb    160
6   Mar    200
7   Mar    240
8   Mar    190

=== COLUMN WISE CONCAT (axis=1) ===
    Name  Age    City  Score
0  Alice   25  Mumbai     88
1    Bob   30   Delhi     92


> **LOGIC:**
> - `axis=0` → same columns, more rows → stacking vertically. Use `ignore_index=True` to reset index.
> - `axis=1` → same rows, more columns → placing side by side. Works only when both have same index.
> - With `axis=1`, different indexes → NaN. Fix with `reset_index(drop=True)` on both before concat.

---
# 📅 SECTION 7 — DateTime Operations
**Dataset: Tesla stock prices** (daily OHLCV data)

In [21]:
# ── 7.1 CONVERTING & EXTRACTING DATETIME ─────────────────

df_dates = pd.DataFrame({'date': [
    '2024-01-01','2024-01-05','2024-02-10',
    '2024-03-15','2024-04-01','2024-04-20'
]})

print('Before conversion:', df_dates['date'].dtype)  # object

# Convert to datetime
df_dates['date'] = pd.to_datetime(df_dates['date'])
print('After conversion:', df_dates['date'].dtype)   # datetime64[ns]

# Extract components
df_dates['year']       = df_dates['date'].dt.year
df_dates['month']      = df_dates['date'].dt.month
df_dates['day']        = df_dates['date'].dt.day
df_dates['dayofweek']  = df_dates['date'].dt.dayofweek  # 0=Mon, 6=Sun
df_dates['day_name']   = df_dates['date'].dt.day_name()
df_dates['quarter']    = df_dates['date'].dt.quarter

print(df_dates)

Before conversion: object
After conversion: datetime64[ns]
        date  year  month  day  dayofweek  day_name  quarter
0 2024-01-01  2024      1    1          0    Monday        1
1 2024-01-05  2024      1    5          4    Friday        1
2 2024-02-10  2024      2   10          5  Saturday        1
3 2024-03-15  2024      3   15          4    Friday        1
4 2024-04-01  2024      4    1          0    Monday        2
5 2024-04-20  2024      4   20          5  Saturday        2


> **LOGIC:** Dates come as strings (`object` type) in raw data. `pd.to_datetime()` converts to `datetime64[ns]`. Once converted, `.dt.` accessor unlocks all date components — same idea as `.str.` for strings.

In [22]:
# ── 7.2 PARSING CUSTOM DATE FORMATS ──────────────────────

from datetime import datetime

# Different date formats in real data
d1 = datetime.strptime('01/15/2024',      '%m/%d/%Y')   # US format
d2 = datetime.strptime('15-Jan-2024',     '%d-%b-%Y')   # with month name
d3 = datetime.strptime('2024-01-15 14:30','%Y-%m-%d %H:%M')  # with time
d4 = datetime.strptime('15/01/2024',      '%d/%m/%Y')   # Indian format

print('US format:     ', d1)
print('Month name:    ', d2)
print('With time:     ', d3)
print('Indian format: ', d4)

print('\n=== FORMAT CODES ===')
codes = pd.DataFrame({
    'Code':    ['%Y','%m','%d','%b','%B','%H','%M','%S'],
    'Meaning': ['4-digit year','2-digit month','2-digit day',
                'Abbreviated month','Full month','Hour (24h)','Minute','Second'],
    'Example': ['2024','01','15','Jan','January','14','30','45']
})
print(codes.to_string(index=False))

US format:      2024-01-15 00:00:00
Month name:     2024-01-15 00:00:00
With time:      2024-01-15 14:30:00
Indian format:  2024-01-15 00:00:00

=== FORMAT CODES ===
Code           Meaning Example
  %Y      4-digit year    2024
  %m     2-digit month      01
  %d       2-digit day      15
  %b Abbreviated month     Jan
  %B        Full month January
  %H        Hour (24h)      14
  %M            Minute      30
  %S            Second      45


In [23]:
# ── 7.3 DATE RANGE & RESAMPLE ────────────────────────────

# Generate date sequences
daily  = pd.date_range(start='2024-01-01', end='2024-01-10', freq='D')
hourly = pd.date_range(start='2024-01-01', periods=24,       freq='H')
monthly= pd.date_range(start='2024-01-01', periods=12,       freq='MS')

print('Daily range:  ', len(daily),   'days')
print('Hourly range: ', len(hourly),  'hours')
print('Monthly range:', len(monthly), 'months')

# Resample — requires datetime as index
# Create sample stock data
np.random.seed(42)
dates  = pd.date_range('2024-01-01', periods=60, freq='D')
stock  = pd.DataFrame({
    'Close':  np.random.uniform(90, 120, 60).round(2),
    'Volume': np.random.randint(1000000, 5000000, 60)
}, index=dates)

print('\n=== MONTHLY AVERAGE CLOSE ===')
print(stock['Close'].resample('M').mean().round(2))

print('\n=== WEEKLY TOTAL VOLUME ===')
print(stock['Volume'].resample('W').sum())

print('\n=== FILTER BY DATE RANGE ===')
jan = stock.loc['2024-01-01':'2024-01-31']
print(f'January rows: {len(jan)}')

Daily range:   10 days
Hourly range:  24 hours
Monthly range: 12 months

=== MONTHLY AVERAGE CLOSE ===
2024-01-31   103.32
2024-02-29   104.78
Freq: ME, Name: Close, dtype: float64

=== WEEKLY TOTAL VOLUME ===
2024-01-07    21861662
2024-01-14    22669433
2024-01-21    19367223
2024-01-28    19981123
2024-02-04    23713145
2024-02-11    24322073
2024-02-18    20765070
2024-02-25    24127044
2024-03-03    11729960
Freq: W-SUN, Name: Volume, dtype: int32

=== FILTER BY DATE RANGE ===
January rows: 31


C:\Users\Naman\AppData\Local\Temp\ipykernel_13556\1775999711.py:5: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  hourly = pd.date_range(start='2024-01-01', periods=24,       freq='H')
C:\Users\Naman\AppData\Local\Temp\ipykernel_13556\1775999711.py:22: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  print(stock['Close'].resample('M').mean().round(2))


> **LOGIC:** `resample()` is like `groupby()` but for time periods. It groups all rows in the same time period and calculates stats.
> - Must set datetime column as index first (`set_index('Date')`)
> - `'M'` = month end, `'MS'` = month start, `'W'` = weekly, `'D'` = daily, `'Q'` = quarterly

---
# 📈 SECTION 8 — Rolling & Cumulative

In [24]:
# ── 8.1 ROLLING WINDOW ───────────────────────────────────

np.random.seed(42)
dates = pd.date_range('2023-01-01', periods=30, freq='D')
stock = pd.DataFrame({
    'Close':  np.random.uniform(90, 120, 30).round(2),
    'High':   np.random.uniform(115, 130, 30).round(2),
    'Low':    np.random.uniform(80, 95, 30).round(2),
    'Volume': np.random.randint(1000000, 5000000, 30)
}, index=dates)

# 7-day moving average — smooths daily noise
stock['MA_7']  = stock['Close'].rolling(window=7).mean().round(2)

# Rolling max and min — price range over last 7 days
stock['7d_high'] = stock['High'].rolling(window=7).max()
stock['7d_low']  = stock['Low'].rolling(window=7).min()

# Rolling std — volatility measure
stock['volatility'] = stock['Close'].rolling(window=7).std().round(2)

print(stock[['Close','MA_7','volatility']].head(12))

print('\n=== DROP NaN ROWS ===')
print(stock[['Close','MA_7']].dropna().head(5))

            Close   MA_7  volatility
2023-01-01 101.24    NaN         NaN
2023-01-02 118.52    NaN         NaN
2023-01-03 111.96    NaN         NaN
2023-01-04 107.96    NaN         NaN
2023-01-05  94.68    NaN         NaN
2023-01-06  94.68    NaN         NaN
2023-01-07  91.74 102.97       10.12
2023-01-08 115.99 105.08       11.18
2023-01-09 108.03 103.58        9.68
2023-01-10 111.24 103.47        9.58
2023-01-11  90.62 101.00       10.43
2023-01-12 119.10 104.49       11.94

=== DROP NaN ROWS ===
            Close   MA_7
2023-01-07  91.74 102.97
2023-01-08 115.99 105.08
2023-01-09 108.03 103.58
2023-01-10 111.24 103.47
2023-01-11  90.62 101.00


> **LOGIC:** Rolling slides a fixed window of N rows down the column.
> ```
> window=7: [1,2,3,4,5,6,7] → calculate → slide down → [2,3,4,5,6,7,8] → calculate
> ```
> - First `N-1` rows are always NaN (window not yet full)
> - `window=7` → first 6 rows NaN. `window=30` → first 29 rows NaN.
> - Use `.dropna()` after rolling if NaN rows are not needed.

In [25]:
# ── 8.2 ROLLING WITH GROUPBY ─────────────────────────────

# Sales data with two products
sales = pd.DataFrame({
    'product': ['A','A','A','A','A','B','B','B','B','B'],
    'day':     [1,2,3,4,5,1,2,3,4,5],
    'revenue': [100,150,200,180,220,300,250,400,350,500]
})

# Rolling sum per product — window resets at each group
sales['rolling_rev'] = (
    sales.groupby('product')['revenue']
         .rolling(window=3)
         .sum()
         .reset_index(level=0, drop=True)
)

print(sales)

  product  day  revenue  rolling_rev
0       A    1      100          NaN
1       A    2      150          NaN
2       A    3      200       450.00
3       A    4      180       530.00
4       A    5      220       600.00
5       B    1      300          NaN
6       B    2      250          NaN
7       B    3      400       950.00
8       B    4      350      1000.00
9       B    5      500      1250.00


In [26]:
# ── 8.3 CUMULATIVE OPERATIONS ────────────────────────────

data = pd.DataFrame({'day': range(1,8), 'sales': [100,150,200,180,220,160,240]})

data['cumulative_sales'] = data['sales'].cumsum()     # running total
data['all_time_high']    = data['sales'].cummax()     # running max
data['all_time_low']     = data['sales'].cummin()     # running min

print(data)

   day  sales  cumulative_sales  all_time_high  all_time_low
0    1    100               100            100           100
1    2    150               250            150           100
2    3    200               450            200           100
3    4    180               630            200           100
4    5    220               850            220           100
5    6    160              1010            220           100
6    7    240              1250            240           100


> **LOGIC:**
> - `rolling()` → fixed window size, slides forward
> - `cumsum()` → window grows from start, never resets
> - Use `cumsum()` for running totals, use `rolling()` for trends

---
# 🏭 SECTION 9 — Complete EDA Workflow
This is how a real data analyst approaches a new dataset from start to finish.

In [27]:
# ── STEP 1: LOAD & INSPECT ────────────────────────────────

df = pd.read_csv('titanic.csv')

print('Shape:', df.shape)
df.info()
print('\nNull %:')
print((df.isnull().sum()/len(df)*100).round(1))
print('\nStats:')
print(df.describe())

Shape: (891, 15)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   survived     891 non-null    int64  
 1   pclass       891 non-null    int64  
 2   sex          891 non-null    object 
 3   age          714 non-null    float64
 4   sibsp        891 non-null    int64  
 5   parch        891 non-null    int64  
 6   fare         891 non-null    float64
 7   embarked     889 non-null    object 
 8   class        891 non-null    object 
 9   who          891 non-null    object 
 10  adult_male   891 non-null    bool   
 11  deck         203 non-null    object 
 12  embark_town  889 non-null    object 
 13  alive        891 non-null    object 
 14  alone        891 non-null    bool   
dtypes: bool(2), float64(2), int64(4), object(7)
memory usage: 92.4+ KB

Null %:
survived       0.00
pclass         0.00
sex            0.00
age           19.90
sib

In [28]:
# ── STEP 2: CLEAN DATA ────────────────────────────────────

# Handle nulls based on column type
df['age'].fillna(df['age'].median(), inplace=True)
df['embarked'].fillna(df['embarked'].mode()[0], inplace=True)
df.drop(columns=['deck'], inplace=True)  # 77% null — useless

# Fix types
df['pclass']   = df['pclass'].astype('category')
df['survived'] = df['survived'].astype(bool)
df['sex']      = df['sex'].astype('category')

# Remove duplicates
df.drop_duplicates(inplace=True)

print('After cleaning:', df.shape)
print(df.isnull().sum())

After cleaning: (775, 14)
survived       0
pclass         0
sex            0
age            0
sibsp          0
parch          0
fare           0
embarked       0
class          0
who            0
adult_male     0
embark_town    2
alive          0
alone          0
dtype: int64


C:\Users\Naman\AppData\Local\Temp\ipykernel_13556\3340659511.py:4: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['age'].fillna(df['age'].median(), inplace=True)
C:\Users\Naman\AppData\Local\Temp\ipykernel_13556\3340659511.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For e

In [29]:
# ── STEP 3: FEATURE ENGINEERING ──────────────────────────

df['family_size'] = df['sibsp'] + df['parch'] + 1
df['is_alone']    = (df['family_size'] == 1)
df['age_group']   = df['age'].apply(lambda x:
    'Child' if x < 18 else ('Senior' if x >= 60 else 'Adult'))
df['fare_tier']   = pd.cut(df['fare'],
    bins=[0,10,50,600], labels=['Low','Mid','High'])

print(df[['age','age_group','family_size','fare_tier']].head(8))

    age age_group  family_size fare_tier
0 22.00     Adult            2       Low
1 38.00     Adult            2      High
2 26.00     Adult            1       Low
3 35.00     Adult            2      High
4 35.00     Adult            1       Low
5 28.00     Adult            1       Low
6 54.00     Adult            1      High
7  2.00     Child            5       Mid


In [30]:
# ── STEP 4: FIND INSIGHTS ─────────────────────────────────

print('=== OVERALL SURVIVAL RATE ===')
print(f"{df['survived'].mean():.1%}")

print('\n=== SURVIVAL BY SEX ===')
print((df.groupby('sex')['survived'].mean()*100).round(1))

print('\n=== SURVIVAL BY CLASS ===')
print((df.groupby('pclass')['survived'].mean()*100).round(1))

print('\n=== SURVIVAL BY AGE GROUP ===')
print((df.groupby('age_group')['survived'].mean()*100).round(1))

print('\n=== TOP 5 HIGHEST FARES ===')
print(df.nlargest(5,'fare')[['sex','pclass','fare','survived']])

print('\n=== CROSSTAB: SURVIVAL RATE BY SEX AND CLASS ===')
print(pd.crosstab(
    df['sex'], df['pclass'],
    values=df['survived'], aggfunc='mean'
).round(2))

print('\n=== CORRELATION ===')
print(df[['age','fare','family_size']].corr().round(2))

=== OVERALL SURVIVAL RATE ===
41.3%

=== SURVIVAL BY SEX ===
sex
female   74.00
male     21.50
Name: survived, dtype: float64

=== SURVIVAL BY CLASS ===
pclass
1   63.30
2   50.60
3   25.90
Name: survived, dtype: float64

=== SURVIVAL BY AGE GROUP ===
age_group
Adult    39.50
Child    54.50
Senior   28.00
Name: survived, dtype: float64

=== TOP 5 HIGHEST FARES ===
        sex pclass   fare  survived
258  female      1 512.33      True
679    male      1 512.33      True
737    male      1 512.33      True
27     male      1 263.00     False
88   female      1 263.00      True

=== CROSSTAB: SURVIVAL RATE BY SEX AND CLASS ===
pclass    1    2    3
sex                  
female 0.97 0.92 0.47
male   0.37 0.18 0.16

=== CORRELATION ===
              age  fare  family_size
age          1.00  0.09        -0.28
fare         0.09  1.00         0.19
family_size -0.28  0.19         1.00


C:\Users\Naman\AppData\Local\Temp\ipykernel_13556\1036649441.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print((df.groupby('sex')['survived'].mean()*100).round(1))
C:\Users\Naman\AppData\Local\Temp\ipykernel_13556\1036649441.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print((df.groupby('pclass')['survived'].mean()*100).round(1))


> **LOGIC:** Standard EDA flow:
> 1. Load → inspect shape, types, nulls
> 2. Clean → handle nulls, fix types, remove duplicates
> 3. Engineer → create useful derived columns
> 4. Analyse → groupby, filter, crosstab to find patterns

---
# 📋 SECTION 10 — Arguments Quick Reference

In [31]:
# ── pd.read_csv() KEY ARGUMENTS ───────────────────────────

# usecols — load only specific columns
df = pd.read_csv('titanic.csv', usecols=['survived','pclass','sex','age','fare'])

# nrows — load only N rows
df = pd.read_csv('titanic.csv', nrows=100)

# skiprows — skip rows at top
df = pd.read_csv('titanic.csv', skiprows=1)

# dtype — force column types on load
df = pd.read_csv('titanic.csv', dtype={'pclass': 'category', 'survived': bool})

# parse_dates — auto-convert date columns
# df = pd.read_csv('data.csv', parse_dates=['date_column'])

# na_values — treat custom values as NaN
df = pd.read_csv('titanic.csv', na_values=['NA','--','N/A','none'])

print('Arguments demo complete')

Arguments demo complete


In [32]:
# ── dropna() KEY ARGUMENTS ────────────────────────────────

df = pd.read_csv('titanic.csv')

# axis — 0=rows, 1=columns
df_no_null_rows = df.dropna(axis=0)  # drop rows with any null
df_no_null_cols = df.dropna(axis=1)  # drop columns with any null

# subset — check nulls only in specific columns
df_age_clean = df.dropna(subset=['age','fare'])

# thresh — keep rows with at least N non-null values
df_thresh = df.dropna(thresh=10)

# how — 'any' (default) or 'all'
df_all = df.dropna(how='all')  # drop only if ALL values are null

print('Original:         ', df.shape)
print('Drop null rows:   ', df_no_null_rows.shape)
print('Drop null cols:   ', df_no_null_cols.shape)
print('Subset age+fare:  ', df_age_clean.shape)
print('Thresh=10:        ', df_thresh.shape)

Original:          (891, 15)
Drop null rows:    (182, 15)
Drop null cols:    (891, 11)
Subset age+fare:   (714, 15)
Thresh=10:         (891, 15)


In [33]:
# ── groupby() KEY ARGUMENTS ───────────────────────────────

df = pd.read_csv('titanic.csv')

# sort=False — don't sort group keys (faster for large data)
print(df.groupby('pclass', sort=False)['fare'].mean())

# dropna=False — include NaN as a group
print('\nWith NaN as group:')
print(df.groupby('embarked', dropna=False)['survived'].count())

# numeric_only — only aggregate numeric columns
print('\nNumeric only:')
print(df.groupby('pclass').mean(numeric_only=True)[['age','fare']].round(2))

pclass
3   13.68
1   84.15
2   20.66
Name: fare, dtype: float64

With NaN as group:
embarked
C      168
Q       77
S      644
NaN      2
Name: survived, dtype: int64

Numeric only:
         age  fare
pclass            
1      38.23 84.15
2      29.88 20.66
3      25.14 13.68


In [34]:
# ── merge() KEY ARGUMENTS ─────────────────────────────────

df1 = pd.DataFrame({'id':[1,2,3], 'val_a':['x','y','z']})
df2 = pd.DataFrame({'id':[1,2,4], 'val_a':['p','q','r'], 'score':[10,20,30]})

# suffixes — rename overlapping columns
result = pd.merge(df1, df2, on='id', how='inner', suffixes=('_left','_right'))
print('=== WITH SUFFIXES ===')
print(result)

# left_on / right_on — different column names
df3 = pd.DataFrame({'emp_id':[1,2,3], 'name':['Alice','Bob','Charlie']})
df4 = pd.DataFrame({'employee_id':[1,2,4], 'salary':[50000,60000,70000]})
result2 = pd.merge(df3, df4, left_on='emp_id', right_on='employee_id', how='left')
print('\n=== DIFFERENT KEY NAMES ===')
print(result2)

=== WITH SUFFIXES ===
   id val_a_left val_a_right  score
0   1          x           p     10
1   2          y           q     20

=== DIFFERENT KEY NAMES ===
   emp_id     name  employee_id   salary
0       1    Alice         1.00 50000.00
1       2      Bob         2.00 60000.00
2       3  Charlie          NaN      NaN


In [35]:
# ── rolling() KEY ARGUMENTS ───────────────────────────────

data = pd.DataFrame({'val': [1,2,None,4,5,6,7,8,9]})

# window — number of rows in window
print('=== window=3 ===')
print(data['val'].rolling(window=3).mean())

# min_periods — calculate even if window not full
print('\n=== min_periods=1 (no NaN at start) ===')
print(data['val'].rolling(window=3, min_periods=1).mean())

# center — use symmetric window around current row
print('\n=== center=True ===')
print(data['val'].rolling(window=3, center=True).mean())

=== window=3 ===
0    NaN
1    NaN
2    NaN
3    NaN
4    NaN
5   5.00
6   6.00
7   7.00
8   8.00
Name: val, dtype: float64

=== min_periods=1 (no NaN at start) ===
0   1.00
1   1.50
2   1.50
3   3.00
4   4.50
5   5.00
6   6.00
7   7.00
8   8.00
Name: val, dtype: float64

=== center=True ===
0    NaN
1    NaN
2    NaN
3    NaN
4   5.00
5   6.00
6   7.00
7   8.00
8    NaN
Name: val, dtype: float64


---
# 🎯 Key Patterns to Remember

| Task | Code |
|------|------|
| Load specific columns | `pd.read_csv(f, usecols=[...])` |
| Null percentage | `(df.isnull().sum()/len(df)*100).round(1)` |
| Fill nulls by column type | `df.fillna({'num_col': df['col'].median(), 'cat_col': df['col'].mode()[0]})` |
| Select object columns | `df.select_dtypes(include=['object'])` |
| Create derived column | `df['col'] = df['x'].apply(lambda x: ...)` |
| Bin numerical column | `pd.cut(df['col'], bins=[...], labels=[...])` |
| Filter + select cols | `df.loc[condition, ['col1','col2']]` |
| Survival rate by group | `df.groupby('col')['survived'].mean()` |
| Multiple agg | `df.groupby('col').agg({'a':'mean','b':['min','max']})` |
| GroupBy to DataFrame | `.size().reset_index(name='count')` |
| Combine monthly files | `pd.concat([jan,feb,mar], ignore_index=True)` |
| Convert date | `pd.to_datetime(df['date'])` |
| Monthly average | `df['col'].resample('M').mean()` |
| Moving average | `df['col'].rolling(window=7).mean()` |
| Running total | `df['col'].cumsum()` |
| Crosstab | `pd.crosstab(df['a'], df['b'], values=df['c'], aggfunc='mean')` |
| Top N rows | `df.nlargest(5, 'col')` |

---
*Pandas Reference Guide — Parth*